# Modello con ResNet18
-Descrivere cosa accade in questo Notebook-

## Import

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import balanced_accuracy_score

## Caricamento Dataset Split

In [ ]:

OUTPUT_CSV = 'train_val_split.csv'

try:
    df = pd.read_csv(OUTPUT_CSV)
    train_df = df[df['split'] == 'train']
    val_df = df[df['split'] == 'val']
    print(f" CSV caricato: {len(train_df)} immagini di Train, {len(val_df)} di Validation.")

except FileNotFoundError:
    raise FileNotFoundError(f" ERRORE: Non trovo il file {OUTPUT_CSV}.")


## Resize Immagini

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

class WasteDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self): return len(self.dataframe)

    def __getitem__(self, idx):

        img_path = self.dataframe.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['label']
        
        if self.transform: image = self.transform(image)

        return image, label



BATCH_SIZE = 32 
train_loader = DataLoader(WasteDataset(train_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=1,pin_memory=True)
val_loader = DataLoader(WasteDataset(val_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=1,pin_memory=True)
print(" Dataloader pronti!")

### Resize con Data Augmentation

In [ ]:
OUTPUT_CSV = 'train_val_split.csv'

try:
    df = pd.read_csv(OUTPUT_CSV)
    train_df = df[df['split'] == 'train']
    val_df = df[df['split'] == 'val']
    print(f" CSV caricato: {len(train_df)} immagini di Train, {len(val_df)} di Validation.")

except FileNotFoundError:
    raise FileNotFoundError(f" ERRORE: Non trovo il file {OUTPUT_CSV}.")

#data loader
print("\n--- FASE 2: Preparazione Tensori ---")

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), # Specchia la foto a destra/sinistra nel 50% dei casi
    transforms.RandomRotation(degrees=15),  # Ruota la foto a caso tra -15 e +15 gradi
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Altera leggermente luce e contrasto
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class WasteDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform


    def __len__(self): return len(self.dataframe)


    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['label']
        
        if self.transform: image = self.transform(image)

        return image, label
    

BATCH_SIZE = 32 
train_loader = DataLoader(WasteDataset(train_df, transform=train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=1,pin_memory=True)
val_loader = DataLoader(WasteDataset(val_df, transform=val_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=1,pin_memory=True)
print(" Dataloader pronti!")


## Caricamento ResNet18, adeguamento e addestramento


### Adam

In [ ]:

# --- FASE 3: Inizializzazione ResNet18 ---

print("\n--- FASE 3: Inizializzazione ResNet18 ---")

# 1. Carichiamo ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 2. Congeliamo i parametri
for param in model.parameters():
    param.requires_grad = False

# 3. Sostituiamo la testa: usiamo '.fc' invece di '.classifier'
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 8)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()

# 4. Ottimizzatore: passiamo '.fc.parameters()'
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
print(f" ResNet18 pronta. Parametri congelati, testa '.fc' modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 10                  # Impostiamo 10 ma grazie all'early stopping , se va in overfitting lo fermiamo prima
PATIENCE = 2                 # Tolleranza di peggioramento sulla Validation Loss
best_val_loss = float('inf') # Memoria del minimo storico della Loss
patience_counter = 0

BEST_MODEL_PATH= 'resnet18_Adam_weights.pth'


### SGD

In [ ]:

# --- FASE 3: Inizializzazione ResNet18 ---

print("\n--- FASE 3: Inizializzazione ResNet18 ---")

# 1. Carichiamo ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 2. Congeliamo i parametri
for param in model.parameters():
    param.requires_grad = False

# 3. Sostituiamo la testa: usiamo '.fc' invece di '.classifier'
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 8)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()

# 4. Ottimizzatore: passiamo '.fc.parameters()'
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
print(f" ResNet18 pronta. Parametri congelati, testa '.fc' modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 10                  # Impostiamo 10 ma grazie all'early stopping , se va in overfitting lo fermiamo prima
PATIENCE = 2                 # Tolleranza di peggioramento sulla Validation Loss
best_val_loss = float('inf') # Memoria del minimo storico della Loss
patience_counter = 0

BEST_MODEL_PATH= 'resnet18_SGD_weights.pth'


### Adam + Data Augumentation

In [ ]:
print("\n--- FASE 3: Inizializzazione ResNet18 ---")

# Carichiamo ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Congeliamo i parametri
for param in model.parameters():
    param.requires_grad = False

# Sostituiamo la testa: usiamo '.fc' invece di '.classifier'
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 8)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()

# 4. Ottimizzatore: passiamo '.fc.parameters()'
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
print(f" ResNet18 pronta. Parametri congelati, testa '.fc' modificata per 8 classi.")
print(f" Acceleratore in uso: {device.type.upper()}")

#addestramento e accuracy
print("\n--- FASE 4: Addestramento ---")

EPOCHS = 10                  # Impostiamo 10 ma grazie all'early stopping , se va in overfitting lo fermiamo prima
PATIENCE = 2                 # Tolleranza di peggioramento sulla Validation Loss
best_val_loss = float('inf') # Memoria del minimo storico della Loss
patience_counter = 0

BEST_MODEL_PATH= 'resnet18_Adam_DataAugmentation_weights.pth'
# Dizionari per salvare la history e plotttare il grafico dell'overfitting per il report


### Ciclo di addestramento

In [ ]:

# Dizionari per salvare la history e plotttare il grafico dell'overfitting per il report
history = {'train_loss': [], 'val_loss': [], 'bal_acc': []}


for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0
    total_train_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):

        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        total_train_loss += loss.item()

        if i % 20 == 19: 

            print(f"  Epoca [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/20:.4f}")

            running_loss = 0.0

            

    print(f" Epoca {epoch+1} conclusa. Calcolo metriche di validazione...")

    

    model.eval()
    all_preds = []
    all_labels = []
    total_val_loss = 0.0 

    with torch.no_grad():

        for inputs, labels in val_loader:

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            val_loss = criterion(outputs, labels)
            total_val_loss += val_loss.item()

            _, predicted = torch.max(outputs.data, 1)
            all_preds.append(predicted.cpu())
            all_labels.append(labels.cpu())          
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # Calcolo medie di fine epoca
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100

       # Salvataggio history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['bal_acc'].append(bal_acc)
            

    print(f"--> Riepilogo Epoca {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Balanced Accuracy: {bal_acc:.2f}%\n")

    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)

        best_preds_memory = all_preds.copy()

        print(f"Val Loss migliorata! Checkpoint ottimale salvato su '{BEST_MODEL_PATH}'\n")
    else:
        patience_counter += 1
        print(f"Val Loss peggiorata o stabile. (Patience: {patience_counter}/{PATIENCE})\n")
        
        if patience_counter >= PATIENCE:
            print(f"EARLY STOPPING INNESCATO ALL'EPOCA {epoch+1}! Interruzione per prevenire Overfitting.")
            break

print("Addestramento concluso!")

## Analisi risultati

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torch


class_names = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']

# Report(Precision, Recall, F1-Score per ogni singola classe)
print("\nCLASSIFICATION REPORT :")
print(classification_report(all_labels, best_preds_memory, target_names=class_names))

# Matrice di Confusione
cm = confusion_matrix(all_labels, best_preds_memory)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Matrice di Confusione - ResNet18 Feature Extraction")
plt.ylabel("Classe Reale (Ground Truth)")
plt.xlabel("Classe Preditta dal Modello")
plt.show()

# Creiamo una lista che va da 1 fino al numero esatto di epoche completate
epoche = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(14, 5))

# GRAFICO 1: LOSS
plt.subplot(1, 2, 1)
# Aggiungiamo 'epoche' come primo parametro (asse X)
plt.plot(epoche, history['train_loss'], label='Train Loss', color='blue', marker='o', linewidth=2)
plt.plot(epoche, history['val_loss'], label='Validation Loss', color='red', marker='o', linewidth=2)
plt.title('Curve di Apprendimento (Evidenza Overfitting)')
plt.xlabel('Epoca')
plt.ylabel('Cross-Entropy Loss')
plt.xticks(epoche) # <-- TRUCCO: Forza i "tic" dell'asse X a essere esattamente 1, 2, 3...
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# GRAFICO 2: ACCURACY
plt.subplot(1, 2, 2)
# Aggiungiamo 'epoche' anche qui
plt.plot(epoche, history['bal_acc'], label='Balanced Accuracy', color='green', marker='s', linewidth=2)
plt.title('Andamento Balanced Accuracy')
plt.xlabel('Epoca')
plt.ylabel('Accuracy (%)')
plt.xticks(epoche) # <-- TRUCCO
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## Analisi immagini

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

################# ANALISI IMMAGINI ##################

def contrastive_error_forensics(model, val_loader, device, class_names, true_class, confused_class, samples_per_row=5):
    """
    Confronta visivamente i campioni falliti con i campioni perfetti della stessa classe.
    """
    model.eval()
    true_idx = class_names.index(true_class)
    conf_idx = class_names.index(confused_class)
    
    mistakes_img, mistakes_conf = [], []
    corrects_img, corrects_conf = [], []
    
    # Costanti di denormalizzazione ImageNet
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs_dev = inputs.to(device)
            outputs = model(inputs_dev)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, 1)
            
            for i in range(len(labels)):
                lbl = labels[i].item()
                prd = preds[i].item()
                cnf = confs[i].item() * 100
                
                # Caso A: Errore mirato (True Class -> Confused Class)
                if lbl == true_idx and prd == conf_idx and len(mistakes_img) < samples_per_row:
                    img = torch.clamp(inputs[i].cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()
                    mistakes_img.append(img)
                    mistakes_conf.append(cnf)
                
                # Caso B: Benchmark di perfezione (True Class -> True Class con altissima sicurezza)
                elif lbl == true_idx and prd == true_idx and cnf > 92.0 and len(corrects_img) < samples_per_row:
                    img = torch.clamp(inputs[i].cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()
                    corrects_img.append(img)
                    corrects_conf.append(cnf)
                    
                if len(mistakes_img) == samples_per_row and len(corrects_img) == samples_per_row:
                    break
            if len(mistakes_img) == samples_per_row and len(corrects_img) == samples_per_row:
                break

    # Creazione della tavola ottica comparativa
    fig, axes = plt.subplots(2, samples_per_row, figsize=(16, 6.5))
    fig.suptitle(f"ANALISI CONTRASTIVA DI CLASSE: [{true_class.upper()}]", fontsize=14, fontweight='bold', y=0.98)
    
    # Render Riga 1: Gli Errori
    for idx in range(samples_per_row):
        ax = axes[0, idx]
        if idx < len(mistakes_img):
            ax.imshow(mistakes_img[idx])
            ax.set_title(f"Scambiato per: {confused_class}\nSicurezza Errore: {mistakes_conf[idx]:.1f}%", color='darkred', fontweight='bold', fontsize=10)
        else:
            ax.text(0.5, 0.5, "Nessun altro\nerrore trovato", ha='center', va='center')
        ax.axis('off')
        if idx == 0: ax.set_ylabel("FALLIMENTI", fontsize=12, fontweight='bold', color='red')

    # Render Riga 2: I Successi
    for idx in range(samples_per_row):
        ax = axes[1, idx]
        if idx < len(corrects_img):
            ax.imshow(corrects_img[idx])
            ax.set_title(f"Predizione Corretta\nSicurezza Rete: {corrects_conf[idx]:.1f}%", color='darkgreen', fontweight='bold', fontsize=10)
        else:
            ax.text(0.5, 0.5, "Campionamento\ninsufficiente", ha='center', va='center')
        ax.axis('off')
        if idx == 0: ax.set_ylabel("BENCHMARK", fontsize=12, fontweight='bold', color='green')

    plt.tight_layout()
    plt.show()

# --- ESEGUI IL TEST CONTRASTIVO ---
# Sostituisci 'Plastic' e 'Glass' con la coppia critica che vedi spuntare dalla tua Matrice di Confusione!
class_names = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']

contrastive_error_forensics(model, val_loader, device, class_names, true_class='Plastic', confused_class='Glass')